# LF replication — GSE113212 bulk microarray vs scRNA pilot signature

Author-independent cross-platform replication test for Arm A. Asks whether the **activated-fibroblast signature** discovered in the GSE294458 single-cell pilot is directionally reproduced in an **independent bulk ligamentum flavum dataset on a different platform** (GSE113212, Agilent microarray; 4 elderly/hypertrophic vs 4 young/non-hypertrophic). This is the disc-arm methodology (agreement across independent platforms) applied to LF, without needing any author to share data.

**Readout:** are the pilot's UP genes coordinately up, and DOWN genes down, in independent bulk hypertrophic LF?

> **Honest caveats.** (1) GSE113212 is **bulk whole tissue** — the fibroblast signal is diluted by other cell types, so a weak result can be dilution rather than a true miss. (2) Its contrast is **age-based** (young vs elderly), a proxy for hypertrophy, not pathology-grading. (3) n = 4 vs 4. Concordance here is supportive, not definitive; discordance is informative but not fatal. Still the strongest public, author-independent replication available.

In [ ]:
# 1. Install
!pip -q install GEOparse scipy pandas numpy matplotlib 2>/dev/null
print("done")

In [ ]:
# 2. Imports
import os, re
import numpy as np, pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
os.makedirs("out", exist_ok=True); os.makedirs("figs", exist_ok=True)
print("ready")

## Step 1 — Download GSE113212 and build a gene-level expression matrix

Pulls the series via GEOparse, assembles a probe x sample matrix, maps Agilent probes to gene symbols (keeping the highest-variance probe per gene), and assigns groups. Group membership is set by GSM accession (with a title-based sanity check).

In [ ]:
# 3. Load + assemble
import GEOparse
gse = GEOparse.get_GEO("GSE113212", destdir="geo", silent=True)

expr = pd.DataFrame({name: g.table.set_index("ID_REF")["VALUE"]
                     for name, g in gse.gsms.items()})
expr = expr.apply(pd.to_numeric, errors="coerce")
print("probe x sample:", expr.shape)

# probe -> gene symbol from the platform annotation
gpl = list(gse.gpls.values())[0]
ann = gpl.table
symcol = next((c for c in ann.columns
               if c.upper() in ("GENE_SYMBOL","GENESYMBOL","SYMBOL","GENE SYMBOL")), None)
idcol = "ID" if "ID" in ann.columns else ann.columns[0]
print("annotation id col:", idcol, "| symbol col:", symcol)
probe2sym = ann.set_index(idcol)[symcol].astype(str)

expr = expr.copy()
expr["sym"] = expr.index.map(probe2sym)
expr = expr[~expr["sym"].isin(["nan","", "--", "NA", "None"])].dropna(subset=["sym"])

# collapse to gene: keep highest-variance probe per symbol
samp_cols = [c for c in expr.columns if c != "sym"]
expr["_var"] = expr[samp_cols].var(axis=1)
gene = (expr.sort_values("_var", ascending=False)
            .drop_duplicates("sym").set_index("sym")[samp_cols])
print("gene x sample:", gene.shape)

# log2 if data look linear (Agilent processed signal)
if np.nanmax(gene.values) > 50 and np.nanmin(gene.values) >= 0:
    gene = np.log2(gene + 1.0)
    print("applied log2 transform")

HYPER = ["GSM3100390","GSM3100391","GSM3100392","GSM3100393"]  # elderly / hypertrophic
CTRL  = ["GSM3100386","GSM3100387","GSM3100388","GSM3100389"]  # young / non-hypertrophic
HYPER = [s for s in HYPER if s in gene.columns]
CTRL  = [s for s in CTRL  if s in gene.columns]
print("hypertrophic:", HYPER)
print("control     :", CTRL)
# sanity check against titles
for s in HYPER + CTRL:
    print(" ", s, "->", gse.gsms[s].metadata.get("title", [""])[0])
assert len(HYPER) >= 2 and len(CTRL) >= 2, "group assignment failed - check GSM ids/titles"

## Step 2 — Differential expression (hypertrophic vs young)

Per-gene Welch t-test with log2 fold-change (hypertrophic minus young). Produces a ranked bulk DE table.

In [ ]:
# 4. Bulk DE
H = gene[HYPER].values; C = gene[CTRL].values
lfc = np.nanmean(H, axis=1) - np.nanmean(C, axis=1)
t, p = stats.ttest_ind(H, C, axis=1, equal_var=False, nan_policy="omit")
bulk = (pd.DataFrame({"gene": gene.index, "log2FC": lfc, "t": t, "pval": p})
          .dropna(subset=["log2FC"]).sort_values("t", ascending=False).reset_index(drop=True))
bulk.to_csv("out/GSE113212_bulk_DE.csv", index=False)
print("bulk DE genes:", len(bulk))
print("top UP in hypertrophic LF:")
print(bulk.head(10)[["gene","log2FC","pval"]].to_string(index=False))

## Step 3 — Replication test vs the scRNA pilot signature

Loads the pilot's activated-fibroblast signature (embedded top genes; falls back to `out/LF_reversal_signature.json` if present). Tests directional concordance: pilot **UP** genes should have **positive** bulk log2FC, **DOWN** genes negative. Reports per-set concordance rates, a one-sided Mann-Whitney (UP fold-changes > DOWN fold-changes), and one-sample sign tests. If the pilot per-gene DE table (`out/LF_STATE_activated_vs_resting.csv`) is present, it also computes the full disc-arm Spearman rank-correlation across shared genes.

In [ ]:
# 5. Replication test
import json
EMBEDDED_UP = ["COL1A2","COL3A1","LUM","ASPN","COL1A1","HTRA1","MFGE8","OGN","TSC22D1",
    "COL5A2","S100A4","ARL6IP5","CD9","DCN","DKK3","CLU","MXRA8","NUPR1","SSPN","SOX5"]
EMBEDDED_DN = ["CTSC","SERPINE1","TM4SF1","MEDAG","CD59","ANGPTL4","APOD","ACKR1",
    "ADAMTS9","IL1RL1","CCL2","AKAP12","CXCL3","TNFRSF12A","CXCL2"]
if os.path.exists("out/LF_reversal_signature.json"):
    s = json.load(open("out/LF_reversal_signature.json"))
    up_sig, dn_sig = s["up"], s["down"]
    print("using saved full signature")
else:
    up_sig, dn_sig = EMBEDDED_UP, EMBEDDED_DN
    print("using embedded pilot top-genes")

lut = bulk.set_index("gene")["log2FC"]
up_lfc = lut.reindex([g for g in up_sig if g in lut.index]).dropna()
dn_lfc = lut.reindex([g for g in dn_sig if g in lut.index]).dropna()
print(f"\nUP genes matched in bulk: {len(up_lfc)}/{len(up_sig)}")
print(f"DOWN genes matched in bulk: {len(dn_lfc)}/{len(dn_sig)}")

conc_up = float((up_lfc > 0).mean()) if len(up_lfc) else float("nan")
conc_dn = float((dn_lfc < 0).mean()) if len(dn_lfc) else float("nan")
print(f"\nUP concordance   (frac log2FC>0): {conc_up:.2f}  | median log2FC {up_lfc.median():+.3f}")
print(f"DOWN concordance (frac log2FC<0): {conc_dn:.2f}  | median log2FC {dn_lfc.median():+.3f}")

if len(up_lfc) and len(dn_lfc):
    U, pmw = stats.mannwhitneyu(up_lfc, dn_lfc, alternative="greater")
    print(f"\nMann-Whitney UP>DOWN log2FC: U={U:.0f}, one-sided p={pmw:.2e}")
if len(up_lfc):
    w_up = stats.wilcoxon(up_lfc, alternative="greater"); print("UP vs 0 (Wilcoxon, greater):   p=%.2e" % w_up.pvalue)
if len(dn_lfc):
    w_dn = stats.wilcoxon(dn_lfc, alternative="less");    print("DOWN vs 0 (Wilcoxon, less):    p=%.2e" % w_dn.pvalue)

# optional: full disc-arm rank correlation if the pilot DE table is present
DEP = "out/LF_STATE_activated_vs_resting.csv"
if os.path.exists(DEP):
    sc_de = pd.read_csv(DEP)
    col = "scores" if "scores" in sc_de.columns else "logfoldchanges"
    m = sc_de[["names", col]].merge(bulk[["gene","t"]], left_on="names", right_on="gene")
    rho, prho = stats.spearmanr(m[col], m["t"])
    print(f"\nFull disc-arm rank-corr (scRNA {col} vs bulk t) over {len(m)} genes: rho={rho:+.3f}, p={prho:.2e}")
else:
    print("\n(For full rank-correlation, also have out/LF_STATE_activated_vs_resting.csv from the pilot in this runtime.)")

## Step 4 — Figure and verdict

In [ ]:
# 6. Plot + verdict
fig, ax = plt.subplots(figsize=(7,5))
data = [up_lfc.values, dn_lfc.values]
parts = ax.boxplot(data, labels=[f"pilot UP\n(n={len(up_lfc)})", f"pilot DOWN\n(n={len(dn_lfc)})"],
                   showmeans=True, widths=0.5)
for i, arr in enumerate(data, start=1):
    x = np.random.default_rng(0).normal(i, 0.05, size=len(arr))
    ax.scatter(x, arr, s=18, alpha=0.7,
               color="#0E8F7E" if i==1 else "#9AA7B4", zorder=3)
ax.axhline(0, color="#888", lw=1, ls="--")
ax.set_ylabel("bulk log2FC (hypertrophic vs young, GSE113212)")
ax.set_title("Pilot scRNA signature tested in independent bulk LF")
fig.tight_layout(); fig.savefig("figs/LF_replication_GSE113212.png", dpi=140)
print("saved figs/LF_replication_GSE113212.png")

ok = (conc_up >= 0.6 and conc_dn >= 0.6 and up_lfc.median() > 0 and dn_lfc.median() < 0)
print("\nVERDICT:",
      "DIRECTIONALLY REPLICATES - activated-fibroblast program is up in independent bulk hypertrophic LF."
      if ok else
      "does NOT cleanly replicate (could be bulk dilution / age-contrast) - inspect per-gene table before concluding.")
print("Reminder: bulk, age-contrast, n=4v4. Supportive evidence, not proof; still one scRNA cohort.")

## Interpretation

- **A clean directional replication** (UP genes up, DOWN genes down, Mann-Whitney significant) would materially strengthen Arm A: a single-cell-discovered fibroblast-activation signature that also holds in an independent platform, despite bulk dilution and an age-based contrast. That is a legitimate discovery-stage result for a provisional + preprint, with the honest ceiling that the single-cell evidence rests on one cohort (GSE294458).
- **A weak or null result** does not kill the hypothesis (bulk whole-tissue dilutes the fibroblast signal, and young-vs-elderly is a rough proxy for hypertrophy), but it means the LF story cannot yet claim cross-platform support and should not advance to filing on this basis.
- Either way, the reversal/candidate step should run on the signature only after this replication readout, and the guard/library upgrades (MoA-based, approved-drug-only) still apply.